# 05 — ScaleSC GPU Harmonized

**Change `DATASET` below to switch datasets.**

ScaleSC is a clustering-first pipeline. No native Wilcoxon DE — validation uses standardized DE.

In [1]:
# === CHANGE THIS TO SWITCH DATASETS ===
DATASET = "mouse_brain_1m"   # or "lung65k" or "mouse_brain_1m"

import json, time, os, shutil
import numpy as np
import pandas as pd
import scanpy as sc
import scalesc as ssc

sc.settings.verbosity = 0
print(f"ScaleSC: {getattr(ssc, '__version__', 'UNKNOWN')}")
print(f"Scanpy: {sc.__version__}")
os.makedirs("write", exist_ok=True)

with open("benchmark_config.json") as f:
    cfg = json.load(f)
gcfg = cfg["global"]
dcfg = cfg["datasets"][DATASET]
prefix = dcfg["pipeline_prefix"]
timings = {}

ScaleSC: 0.1.0
Scanpy: 1.12


## Prepare ScaleSC-compatible input

ScaleSC needs raw counts in `.X`. Our canonical h5ad stores normalized data in `.X`
and counts in `.layers['counts']`. Fix that here.

Also clean the folder so only ONE h5ad exists (ScaleSC loads all files in the folder).

In [2]:
scalesc_dir = f"write/scalesc_input_{prefix}"

# Clean folder completely
if os.path.exists(scalesc_dir):
    shutil.rmtree(scalesc_dir)
os.makedirs(scalesc_dir)

# Load canonical, put raw counts in .X
adata_tmp = sc.read_h5ad(dcfg["canonical_h5ad"])
adata_tmp.X = adata_tmp.layers["counts"].copy()
del adata_tmp.layers["counts"]

# For mouse_brain_1m: subset to 1M cells (int32 sparse ceiling)
N_CELLS = 1_000_000
full_n_obs = adata_tmp.n_obs
if adata_tmp.n_obs > N_CELLS:
    print(f"Subsetting to {N_CELLS:,} cells (from {adata_tmp.n_obs:,})")
    adata_tmp = adata_tmp[:N_CELLS, :].copy()

dest = os.path.join(scalesc_dir, f"{prefix}_for_scalesc.h5ad")
adata_tmp.write_h5ad(dest)
print(f"Saved: {dest} ({adata_tmp.n_obs:,} cells x {adata_tmp.n_vars:,} genes)")
del adata_tmp

Subsetting to 1,000,000 cells (from 1,153,539)
Saved: write/scalesc_input_mouse_1m/mouse_1m_for_scalesc.h5ad (1,000,000 cells x 22,788 genes)


## Initialize ScaleSC

In [3]:
t0 = time.time()
scalesc = ssc.ScaleSC(
    data_dir=scalesc_dir,
    max_cell_batch=1e5,
    preload_on_cpu=True,
    preload_on_gpu=True,
    save_raw_counts=False,
    save_norm_counts=False,
    output_dir="write",
)
timings["init"] = time.time() - t0

# Fix broken reader shape
if scalesc.reader.shape[0] < 0:
    n, m = scalesc.adata.shape
    scalesc.reader.n_cell = n
    scalesc.reader.n_cell_origin = n
    scalesc.reader.n_gene = m
    scalesc.reader.n_gene_origin = m
    print(f"Fixed reader shape: {scalesc.reader.shape}")

print(f"ScaleSC initialized in {timings['init']:.1f}s")
print(f"adata shape: {scalesc.adata.shape}")
print(f"reader shape: {scalesc.reader.shape}")

Fixed reader shape: (1000000, 22788)
ScaleSC initialized in 14.6s
adata shape: (1000000, 22788)
reader shape: (1000000, 22788)


## Normalize + log1p

In [4]:
t0 = time.time()
scalesc.normalize_log1p(target_sum=gcfg["target_sum"])
timings["normalize_log1p"] = time.time() - t0
print(f"Normalize + log1p: {timings['normalize_log1p']:.1f}s")

Normalize + log1p: 1.4s


## HVG

In [5]:
t0 = time.time()
scalesc.highly_variable_genes(n_top_genes=dcfg["n_top_genes"], method="seurat_v3")
timings["hvg"] = time.time() - t0
n_hvg = scalesc.adata.var["highly_variable"].sum()
print(f"HVG ({n_hvg} genes): {timings['hvg']:.1f}s")

HVG (5000 genes): 0.3s


## PCA

In [6]:
t0 = time.time()
scalesc.pca(n_components=gcfg["pca_n_comps"], hvg_var="highly_variable")
timings["pca"] = time.time() - t0
print(f"PCA: {timings['pca']:.1f}s")

PCA: 8.5s


## Neighbors

ScaleSC defaults to `cagra` (approximate GPU graph search).

In [7]:
t0 = time.time()
scalesc.neighbors(
    n_neighbors=dcfg["n_neighbors"],
    n_pcs=dcfg["neighbors_n_pcs"],
    use_rep="X_pca",
    algorithm="cagra",
)
timings["neighbors"] = time.time() - t0
print(f"Neighbors (cagra): {timings['neighbors']:.1f}s")

Neighbors (cagra): 7.4s


## Leiden

In [8]:
t0 = time.time()
scalesc.leiden(
    resolution=dcfg["leiden_resolution"],
    random_state=gcfg["random_state"],
)
timings["leiden"] = time.time() - t0
n_clusters = scalesc.adata.obs["leiden"].nunique()
print(f"Leiden ({n_clusters} clusters): {timings['leiden']:.1f}s")

Leiden (27 clusters): 3.2s


## UMAP

In [9]:
t0 = time.time()
scalesc.umap(random_state=gcfg["random_state"])
timings["umap"] = time.time() - t0
print(f"UMAP: {timings['umap']:.1f}s")

UMAP: 3.3s


## Save outputs

In [10]:
out = f"write/{prefix}_scalesc_gpu_harmonized"
adata = scalesc.adata

# Clusters
pd.DataFrame({
    "barcode": adata.obs_names.astype(str),
    "leiden": adata.obs["leiden"].astype(str).values,
}).to_csv(f"{out}_clusters.csv", index=False)

# UMAP
if "X_umap" in adata.obsm:
    pd.DataFrame(
        adata.obsm["X_umap"],
        index=adata.obs_names.astype(str),
        columns=["UMAP_1", "UMAP_2"],
    ).reset_index(names="barcode").to_csv(f"{out}_umap.csv", index=False)

# h5ad
t0 = time.time()
adata.write_h5ad(f"{out}.h5ad", compression="gzip")
timings["save_h5ad"] = time.time() - t0

# Spec
spec = {
    "pipeline": "scalesc_gpu_harmonized",
    "dataset": dcfg["name"],
    "scalesc_version": getattr(ssc, "__version__", "UNKNOWN"),
    "neighbors_algorithm": "cagra",
    "native_wilcoxon_de": False,
    "reader_shape_patched": True,
    "cell_subset": {
        "n_cells_used": int(adata.n_obs),
        "n_cells_total": full_n_obs,
    },
    "timings_seconds": timings,
    "parameters": {
        "target_sum": gcfg["target_sum"],
        "n_top_genes": dcfg["n_top_genes"],
        "pca_n_comps": gcfg["pca_n_comps"],
        "n_neighbors": dcfg["n_neighbors"],
        "neighbors_n_pcs": dcfg["neighbors_n_pcs"],
        "neighbors_algorithm": "cagra",
        "leiden_resolution": dcfg["leiden_resolution"],
        "random_state": gcfg["random_state"],
    },
    "results": {
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "n_hvg": int(n_hvg),
        "n_clusters": n_clusters,
    },
}
with open(f"{out}_spec.json", "w") as f:
    json.dump(spec, f, indent=2)

print("\n=== Timings ===")
total = 0
for step, t in timings.items():
    print(f"  {step:20s}: {t:8.1f}s")
    total += t
print(f"  {'TOTAL':20s}: {total:8.1f}s ({total/60:.1f} min)")
print(f"\nCells: {adata.n_obs:,}")
print(f"Clusters: {n_clusters}")
print(f"\nNote: No native DE. Run validation notebook for standardized DE.")

... storing 'leiden' as categorical



=== Timings ===
  init                :     14.6s
  normalize_log1p     :      1.4s
  hvg                 :      0.3s
  pca                 :      8.5s
  neighbors           :      7.4s
  leiden              :      3.2s
  umap                :      3.3s
  save_h5ad           :     13.3s
  TOTAL               :     51.9s (0.9 min)

Cells: 1,000,000
Clusters: 27

Note: No native DE. Run validation notebook for standardized DE.
